# WFGY Seven Millennium Problems Speedrun · Colab Runner

Upload `Seven_Millennium_Problems_effective_layer_reproducible_speedrun.zip`, then run all cells.

This notebook checks SHA256, extracts the package, runs the public speedrun runner, runs C03 writeback validation, runs C04 two-way statement validation, measures runtime, prints terminal-style output, and downloads a small results ZIP.


In [ ]:
# Cell 1 — Upload the speedrun ZIP and verify SHA256
from google.colab import files
from pathlib import Path
import hashlib, os, zipfile, json, time, subprocess, textwrap, shutil

EXPECTED_SHA256 = "FA767D99BFCD55041FDBCC86C5BD62F781DF9B39C617E5B28E3BD461C3FFE042"
EXPECTED_FILENAME_HINT = "Seven_Millennium_Problems_effective_layer_reproducible_speedrun.zip"

print("🚀 WFGY Seven Millennium Problems Speedrun · Colab Runner")
print("📦 Please upload:", EXPECTED_FILENAME_HINT)

uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file uploaded.")

zip_path = Path(next(iter(uploaded.keys()))).resolve()
print("\nUploaded file:", zip_path.name)
print("File size:", zip_path.stat().st_size, "bytes")

h = hashlib.sha256()
with zip_path.open("rb") as f:
    for chunk in iter(lambda: f.read(1024 * 1024), b""):
        h.update(chunk)
actual_sha = h.hexdigest().upper()
print("SHA256:", actual_sha)

if actual_sha != EXPECTED_SHA256:
    print("⚠️ SHA256 mismatch.")
    print("Expected:", EXPECTED_SHA256)
    print("Actual:  ", actual_sha)
    raise SystemExit("Checksum mismatch. Stop before validation.")

print("✅ SHA256 PASS")


In [ ]:
# Cell 2 — Extract package and locate root folder
WORKDIR = Path("/content/wfgy_speedrun_work")
if WORKDIR.exists():
    shutil.rmtree(WORKDIR)
WORKDIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(WORKDIR)

# Find package root by locating the public runner.
runner_candidates = list(WORKDIR.rglob("tools/run_public_rpg_speedrun.py"))
if not runner_candidates:
    raise SystemExit("Cannot find tools/run_public_rpg_speedrun.py inside ZIP.")

PACKAGE_ROOT = runner_candidates[0].parents[1]
print("✅ Extracted package root:", PACKAGE_ROOT)
print("Top-level package folders:")
for p in sorted(PACKAGE_ROOT.iterdir()):
    if p.is_dir():
        print(" -", p.name)


In [ ]:
# Cell 3 — Run public speedrun, C03 validator, and C04 validator with timing
import datetime

def run_cmd(label, cmd, cwd):
    print("\n" + "=" * 88)
    print(f"▶ {label}")
    print("Command:", " ".join(cmd))
    print("Start:", datetime.datetime.now().isoformat(timespec="seconds"))
    t0 = time.perf_counter()
    proc = subprocess.run(cmd, cwd=str(cwd), text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    elapsed = time.perf_counter() - t0
    print(proc.stdout)
    print(f"⏱️ {label} elapsed_seconds: {elapsed:.3f}")
    print("Return code:", proc.returncode)
    if proc.returncode != 0:
        raise SystemExit(f"{label} failed")
    return {"label": label, "cmd": cmd, "elapsed_seconds": elapsed, "returncode": proc.returncode, "stdout": proc.stdout}

results = []
wall_t0 = time.perf_counter()

results.append(run_cmd(
    "Public RPG Speedrun Runner",
    ["python", "tools/run_public_rpg_speedrun.py", "--root", ".", "--no-emoji", "--summary-json", "PUBLIC_RPG_RUN_SUMMARY.json"],
    PACKAGE_ROOT,
))

results.append(run_cmd(
    "C03 Original Theorem Writeback Validator",
    ["python", "tools/validate_c03_writeback.py", "--root", "."],
    PACKAGE_ROOT,
))

results.append(run_cmd(
    "C04 Two-Way Official Statement Validation Data Validator",
    ["python", "tools/validate_two_way_statement_validation.py", "--root", "."],
    PACKAGE_ROOT,
))

wall_elapsed = time.perf_counter() - wall_t0
print("\n" + "=" * 88)
print("🏁 COLAB SPEEDRUN VALIDATION COMPLETE")
print(f"Total wall-clock elapsed_seconds: {wall_elapsed:.3f}")


In [ ]:
# Cell 4 — Print compact final summary and package result ZIP for download
summary_path = PACKAGE_ROOT / "PUBLIC_RPG_RUN_SUMMARY.json"
summary = {}
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding="utf-8"))

final_report = {
    "uploaded_file": zip_path.name,
    "sha256": actual_sha,
    "sha256_pass": actual_sha == EXPECTED_SHA256,
    "package_root": str(PACKAGE_ROOT),
    "total_elapsed_seconds_from_cells": sum(r["elapsed_seconds"] for r in results),
    "commands": [{"label": r["label"], "elapsed_seconds": r["elapsed_seconds"], "returncode": r["returncode"]} for r in results],
    "public_rpg_summary": summary,
    "final_colab_status": "PASS",
}

out_dir = Path("/content/wfgy_speedrun_results")
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

(out_dir / "COLAB_VALIDATION_REPORT.json").write_text(json.dumps(final_report, indent=2, ensure_ascii=False), encoding="utf-8")

log_text = []
for r in results:
    log_text.append("=" * 88)
    log_text.append(r["label"])
    log_text.append(f"elapsed_seconds: {r['elapsed_seconds']:.3f}")
    log_text.append(r["stdout"])
(out_dir / "COLAB_TERMINAL_OUTPUT.txt").write_text("\n".join(log_text), encoding="utf-8")

# Copy package-generated summary if present.
if summary_path.exists():
    shutil.copy2(summary_path, out_dir / "PUBLIC_RPG_RUN_SUMMARY.json")

print("✅ Final Colab status: PASS")
print("SHA256:", actual_sha)
print("Total command elapsed seconds:", f"{final_report['total_elapsed_seconds_from_cells']:.3f}")

# Show key lines from summary when available.
if summary:
    print("\nPublic RPG summary keys:")
    for key in ["all_bosses_passed", "total_open_strict_debt_cells", "final_status", "c03_writeback_status", "two_way_statement_validation_status", "official_statement_intake_debt", "two_way_statement_validation_debt"]:
        if key in summary:
            print(f"{key}: {summary[key]}")

result_zip = shutil.make_archive("/content/WFGY_Seven_Millennium_Speedrun_Colab_Results", "zip", out_dir)
print("\n📦 Result ZIP:", result_zip)
files.download(result_zip)
